# Synthetic data from knowledge

References:

* [Nemotron-CC](https://arxiv.org/abs/2412.02595v2)
* [BeyondWeb](https://blog.datologyai.com/beyondweb/)
* [Kimi K2](https://arxiv.org/abs/2507.20534)

In [1]:
from pathlib import Path

OUTPUT_DIR = Path("../output/synthetic/v2")
INPUT_DIR = Path("../output/wiki/fandom/crawl")

## Rephraser model

In [2]:
import dspy
from pydantic import BaseModel


class QuestionAnswerPair(BaseModel):
    """A question and its corresponding answer."""
    question: str
    answer: str


class QAGenerator(dspy.Signature):
    """Read the text, ask questions and answer them.

Follow these instructions:
1. Ask diverse questions that require different cognitive skills or cover different aspects of the text.
2. Ask questions in various forms such as:
- Yes/No questions that require determining whether a statement is true or false.
- Open-ended questions that begin with words like what, how, when, where, why and who.
- Multi-choice questions that offers two or more options to choose from. Include the options in the question.
- Comparison questions that compare two quantities or objects and determine the relationship between them.
- Reading comprehension questions that test the ability to understand and analyze the text.
- Problem-solving questions that test the ability to solve mathematical, physical, or logical problems.
- Long-form questions that require detailed answers or explanations.
3. Use the document summary and previous question-answer pairs as context.
- Do not repeat questions that have already been asked, or that are too similar to previously asked questions.
- New questions should be relevant only to the current document segment.
- New questions should follow naturally from the previous questions and answers.
3. Focus on asking questions about factual information, important knowledge, or concrete details in the text.
4. Write questions and answers using clear and concise language.
5. Use plain text. Do not use Markdown."""

    summary: str = dspy.InputField(
        desc="A brief summary of the whole document, for context."
    )
    previous_qa_pairs: list[QuestionAnswerPair] = dspy.InputField(
        desc="A list of previously asked question-answer pairs for the previous segment of the document."
    )
    document_segment: str = dspy.InputField(
        desc="The document segment to ask questions about."
    )
    question_answers: list[QuestionAnswerPair] = dspy.OutputField(
        desc="A list of question-answer pairs.",
        max_length=8,
    )


class SyntheticQAGenerator(dspy.Module):
    """Generate synthetic question-answer pairs from text segments."""

    def __init__(self):
        self.summarizer = dspy.Predict(dspy.make_signature(
            signature="text -> summary",
            instructions="Summarize the following text in a concise manner.",
        ))
        self.qa_generator = dspy.ChainOfThought(QAGenerator)

    def forward(self, doc: str, segments: list[str]) -> dspy.Prediction:
        summary = self.summarizer(text=doc).summary

        res = []
        prev = []
        for segment in segments:
            qa_pairs = self.qa_generator(
                summary=summary,
                previous_qa_pairs=prev,
                document_segment=segment
            ).question_answers
            res.extend(qa_pairs)
            prev = qa_pairs

        return dspy.Prediction(summary=summary, qa_pairs=res)

## Chunking

In [3]:
from markdown_it import MarkdownIt


def split_markdown(md_text: str) -> list[str]:
    md = MarkdownIt()
    tokens = md.parse(md_text)

    chunks = []
    current = []
    for i, tok in enumerate(tokens):
        if tok.type == "heading_open":
            # flush previous chunk
            if current:
                chunks.append("".join(current))
                current = []
        # reconstruct token text
        if tok.type == "inline":
            current.append(tok.content + "\n")
        elif tok.type.endswith("_open") or tok.type.endswith("_close"):
            # keep markup like ###, code fences, list markers
            current.append(tok.markup + "\n")
        elif tok.type == "fence":
            current.append(f"{tok.markup}{tok.info}\n{tok.content}{tok.markup}\n")
        elif tok.content:
            current.append(tok.content + "\n")

    if current:
        chunks.append("".join(current))

    return chunks

In [4]:
with open("../output/wiki/issues.md") as f:
    issues_md = f.read()

chunks = split_markdown(issues_md)

In [5]:
with open("../output/wiki/uno.md") as f:
    uno_md = f.read()

tokens = MarkdownIt().parse(uno_md)
set(token.type for token in tokens)

{'bullet_list_close',
 'bullet_list_open',
 'heading_close',
 'heading_open',
 'inline',
 'list_item_close',
 'list_item_open',
 'paragraph_close',
 'paragraph_open'}

In [38]:
from dataclasses import dataclass, field
from collections.abc import Iterator
from markdown_it import MarkdownIt


@dataclass
class Heading:
    level: int
    title: str

    @classmethod
    def from_token(cls, tag: str, content: str) -> "Heading":
        level = int(tag[1:])
        return Heading(level=level, title=content)


@dataclass
class Headings:
    lst: list[Heading] = field(default_factory=list)

    def __str__(self) -> str:
        return "\n\n".join(
            f"{'#' * h.level} {h.title}"
            for h in self.lst
        ) + "\n"

    def add(self, heading: Heading):
        # Remove any headings at the same or deeper level
        while self.lst and self.lst[-1].level >= heading.level:
            self.lst.pop()
        self.lst.append(heading)

    def merge(self, other: "Headings") -> "Headings":
        """Merge two Headings lists, keeping the deepest headings."""
        res = Headings(lst=self.lst.copy())
        for heading in other.lst:
            res.add(heading)
        return res


@dataclass
class Segment:
    """A text segment with all the headings in it."""
    headings: Headings = field(default_factory=Headings)
    text: str = ""


def iter_markdown_segments(txt: str) -> Iterator[Segment]:
    lines = txt.splitlines(keepends=True)
    tokens = MarkdownIt().parse(txt)
    segment_headings: Headings = Headings()
    segment_begin = 0
    i = 0

    while i < len(tokens):
        match tokens[i].type:
            case "heading_open":
                h = Heading.from_token(
                    tag=tokens[i].tag,
                    content=tokens[i + 1].content
                )
                segment_headings.add(h)
                i += 2

            case "paragraph_open":
                _, end = tokens[i].map or []
                # If the inline element is empty, skip it
                if tokens[i + 1].content.strip() == "":
                    i += 3
                    continue

                text = "".join(lines[segment_begin:end])
                yield Segment(
                    headings=segment_headings,
                    text=text
                )
                i += 2
                segment_begin = end
                segment_headings = Headings()

            case "fence":
                _, end = tokens[i].map or []
                text = "".join(lines[segment_begin:end])
                yield Segment(
                    headings=segment_headings,
                    text=text
                )
                i += 1
                segment_begin = end
                segment_headings = Headings()

            case _:
                i += 1


# Tests
md = """# This is the title

Body text with **bold** and _italic_ and a [link](http://example.com).

## This is a subheading

### This is a sub-subheading

Another paragraph, with:

- a bullet point
- another bullet point
  - a nested bullet point
- last bullet point

```java
some code
```

A paragraph
"""

[
    s
    for s in iter_markdown_segments(md)
]

[Segment(headings=Headings(lst=[Heading(level=1, title='This is the title')]), text='# This is the title\n\nBody text with **bold** and _italic_ and a [link](http://example.com).\n'),
 Segment(headings=Headings(lst=[Heading(level=2, title='This is a subheading'), Heading(level=3, title='This is a sub-subheading')]), text='\n## This is a subheading\n\n### This is a sub-subheading\n\nAnother paragraph, with:\n'),
 Segment(headings=Headings(lst=[]), text='\n- a bullet point\n'),
 Segment(headings=Headings(lst=[]), text='- another bullet point\n'),
 Segment(headings=Headings(lst=[]), text='  - a nested bullet point\n'),
 Segment(headings=Headings(lst=[]), text='- last bullet point\n'),
 Segment(headings=Headings(lst=[]), text='\n```java\nsome code\n```\n'),
 Segment(headings=Headings(lst=[]), text='\nA paragraph\n')]

In [46]:
def compact_markdown_segments(
        seq: Iterator[Segment],
        min_length: int = 256,
        max_length: int = 2048,
) -> list[Segment]:
    compacted = []
    curr = Segment()

    for seg in seq:
        if (
            len(curr.text) < min_length
            or len(curr.text) + len(seg.text) <= max_length
        ):
            # merge segments if the current is too short,
            # or if the combined length is within limits
            curr.text += seg.text
            curr.headings = curr.headings.merge(seg.headings)
        else:
            compacted.append(curr)
            curr = Segment(
                headings=curr.headings.merge(seg.headings),
                text=str(curr.headings) + seg.text
            )

    if len(curr.text) > min_length:
        compacted.append(curr)

    return compacted


def markdown_segments(
        txt: str,
        min_length: int = 256,
        max_length: int = 2048
) -> list[str]:
    return [
        seg.text
        for seg in compact_markdown_segments(
            iter_markdown_segments(txt), min_length, max_length
        )
    ]


markdown_segments(md, min_length=10, max_length=128)

['# This is the title\n\nBody text with **bold** and _italic_ and a [link](http://example.com).\n',
 '# This is the title\n\n## This is a subheading\n\n### This is a sub-subheading\n\nAnother paragraph, with:\n\n- a bullet point\n',
 '# This is the title\n\n## This is a subheading\n\n### This is a sub-subheading\n- another bullet point\n  - a nested bullet point\n',
 '# This is the title\n\n## This is a subheading\n\n### This is a sub-subheading\n- last bullet point\n\n```java\nsome code\n```\n',
 '# This is the title\n\n## This is a subheading\n\n### This is a sub-subheading\n\nA paragraph\n']

In [47]:
with open("../output/wiki/uno.md") as f:
    uno_md = f.read()

uno_seg = markdown_segments(uno_md, min_length=128, max_length=1024)
uno_seg

["# Uno (personaggio)\n\nUno è uno dei personaggi principali della serie a fumetti Disney Italia di PK (PK - Paperinik New Adventures, PK², PK - Pikappa e PK - Nuova era).\n\n## Biografia\n\nSi tratta di un'intelligenza artificiale creata da Everett Ducklair. È in grado di controllare in ogni sua parte la Ducklair Tower, che ne costituisce il corpo centrale (tanto da presentarsi a Paperinik dicendo di essere a tutti gli effetti il palazzo), di svolgere calcoli di estrema difficoltà e di introdursi grazie alle sue superiori capacità nelle reti informatiche convenzionali più protette. Controlla anche la grande parte dei marchingegni costruiti da Ducklair.\n",
 "# Uno (personaggio)\n\n## Biografia\n\nÈ il più valido aiuto di Paperinik, di cui conosce l'identità segreta. L'aspetto con cui si presenta agli umani è molto simile al viso di Everett Ducklair, solitamente presentato in forma di ologramma all'interno di una sfera verde. Il suo acerrimo nemico è Due, la crudele intelligenza artifi